## Imports

In [1]:
import random
import torch
from torch import nn
from torch.amp import GradScaler
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from sklearn import preprocessing
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from matplotlib import colormaps
import pandas as pd
import copy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

seed = 7
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

scaler = GradScaler(device=device)

# because we do NOT want to see 50 plots in our notebook
plt.ioff()

# for mapping labels
label_encoder = preprocessing.LabelEncoder()

# toggle this to retrain the probes
trainToggle = True

Using device: cuda


## constants

In [2]:
# CONSTANTS --------------------------------------------------------------------------

phone_map = {'aa': 0, 'ae': 1, 'ah': 2, 'ao': 3, 'aw': 4, 'ax': 5, 'ax-h': 6, 'axr': 7, 
'ay': 8, 'b': 9, 'bcl': 10, 'ch': 11, 'd': 12, 'dcl': 13, 'dh': 14, 'dx': 15, 
'eh': 16, 'el': 17, 'em': 18, 'en': 19, 'eng': 20, 'epi': 21, 'er': 22, 'ey': 23, 
'f': 24, 'g': 25, 'gcl': 26, 'h#': 27, 'hh': 28, 'hv': 29, 'ih': 30, 'ix': 31, 
'iy': 32, 'jh': 33, 'k': 34, 'kcl': 35, 'l': 36, 'm': 37, 'n': 38, 'ng': 39, 
'nx': 40, 'ow': 41, 'oy': 42, 'p': 43, 'pau': 44, 'pcl': 45, 'q': 46, 'r': 47, 
's': 48, 'sh': 49, 't': 50, 'tcl': 51, 'th': 52, 'uh': 53, 'uw': 54, 'ux': 55, 
'v': 56, 'w': 57, 'y': 58, 'z': 59, 'zh': 60}

# from TIMIT PHONCODE.DOC
moa_map = {
    'stop': {'b', 'd', 'g', 'p', 't', 'k', 'dx', 'q'},
    'affricate': {'jh', 'ch'},
    'fricative': {'s', 'sh', 'z', 'zh', 'f', 'th', 'v', 'dh'},
    'nasal': {'m', 'n', 'ng', 'em', 'en', 'eng', 'nx'},
    'approximant': {'l', 'r', 'w', 'y', 'hh', 'hv', 'el'},
    'vowel': {'iy', 'ih', 'eh', 'ey', 'ae', 'aa', 'aw', 'ay', 'ah', 'ao', 'oy', 'ow', 'uh', 'uw', 'ux', 'er', 'ax', 'ix', 'axr', 'ax-h'},
    'none': {'h#', 'pau', 'epi', '1', '2'}
}

# voiced_map = {
#     'voiced': {'b', 'd', 'g', 'dx', 'jh', 'z', 'zh', 'v', 'dh', 'm', 'n', 'ng', 'em', 'en', 'eng', 'nx', 'l', 'r', 'w', 'y', 'el', 'iy', 'ih', 'eh', 'ey', 'ae', 'aa', 'aw', 'ay', 'ah', 'ao', 'oy', 'ow', 'uh', 'uw', 'ux', 'er', 'ax', 'ix', 'axr', 'ax-h'},
#     'unvoiced': {'p', 't', 'k', 'q', 'ch', 's', 'sh', 'f', 'th', 'hh', 'hv'}
# }

# vowel_height_map = {
#     'high': {'iy', 'ih', 'ey', 'uh','uw','ux','ix'},
#     'mid': {'eh', 'ae', 'ah', 'ao', 'er', 'ax', 'axr', 'ax-h'},
#     'low': {'aa', },
#     'diphthong': {'aw', 'ay', 'oy', 'ow'}
# }

# invert the map so we can convert from phones to manner of articulation
phone_to_moa_map = {}
for moa, phone_set in moa_map.items():
    for phone in phone_set:
        phone_to_moa_map[phone] = moa

def phone_to_moa(phone):
    if phone in phone_to_moa_map.keys():
        return phone_to_moa_map[phone]
    return 'none'

# # invert the map so we can convert from phones to voicedness
# phone_to_voiced_map = {}
# for voicedness, phone_set in moa_map.items():
#     for phone in phone_set:
#         phone_to_voiced_map[phone] = voicedness

# def phone_to_voiced(phone):
#     if phone in phone_to_voiced_map.keys():
#         return phone_to_voiced_map[phone]
#     return 'none'

### dataset and probe definitions

In [3]:
# dataset definition
class AudioDataset(Dataset):
    def __init__(self, x, y):
        self.x         = x
        self.y         = y

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return len(self.x)

# define probe architecture
class ProbeNet(nn.Module):
    def __init__(self, embedding_dim, encoder="", encoderLayer=-1):
        super(ProbeNet, self).__init__()

        self.encoder = encoder
        self.encoderLayer = encoderLayer

        self.layers = nn.Sequential(
            nn.Linear(embedding_dim, 200),
            nn.ReLU(),
            # nn.Dropout(0.2),
            # nn.Linear(200, 61) # 61 phones
            nn.Linear(200, 6) # 6 manners of articulation
        )
    
    def forward(self, x):
        x = self.layers(x)
        return x

## helper functions

In [4]:
def data_setup(embeddings, labels, validation_frac=0.2, labels_generated=True):
    # go from raw phones to MOA labels
    labels = list(map(phone_to_moa, labels))

    # filter out none values
    new_embeddings = []
    new_labels = []

    for i in range(len(labels)):
        if labels[i] != 'none':
            new_embeddings.append(embeddings[i])
            new_labels.append(labels[i])

    # if this is the first time generating the label encoder:
    if not labels_generated:
        # convert our phone labels into integers so we can fit to them
        labels_types = label_encoder.fit_transform(new_labels)
    else:
        # just reue already generated ones
        labels_types = label_encoder.transform(new_labels)
    
    

    print("records: " + str(len(new_embeddings)))

    dataset = AudioDataset(new_embeddings, labels_types)
    validation_size = int(len(dataset) * validation_frac)
    train_size = len(dataset) - validation_size

    train_set, validation_set = torch.utils.data.random_split(
        dataset,
        [train_size, validation_size],
        generator=torch.Generator().manual_seed(seed)
    )

    train_loader = DataLoader(
        dataset=train_set,
        batch_size=128,
        shuffle=True,
        num_workers=0
    )

    validation_loader = DataLoader(
        dataset=validation_set,
        batch_size=128,
        shuffle=False,
        num_workers=0
    )

    print('train batches: ' + str(len(train_loader)))
    print('validate batches: ' + str(len(validation_loader)))

    return train_loader, validation_loader

def train(probe, train_loader, validation_loader, writer, epochs, criterion, optimizer, note):

    best_validation_loss = float("inf")
    best_model = None
    epochs_without_improvement = 0

    for epoch in tqdm(range(1, epochs + 1),
            "Training " + probe.encoder + " layer " + str(probe.encoderLayer),
            epochs
    ):
        # trainnig ---------------------------------------------------
        probe.train()

        correct = 0
        total = 0
        training_loss = 0.0

        for data in train_loader:
            inputs = data[0][probe.encoderLayer].to(device).float()
            targets = data[1].to(device)

            # for slice in data:
            #     print(slice[probe.encoderLayer].shape)
            # print(data[probe.encoderLayer].shape)

            outputs = probe(inputs)

            loss = criterion(outputs, targets)

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            training_loss += loss.item()
            predicted = torch.argmax(outputs, dim=1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
        
        train_accuracy = 100.*correct / total
        writer.add_scalar(note + "/train_accuracy", train_accuracy, epoch)

        # validation ------------------------------------------------
        probe.eval()
        validation_loss = 0.0
        validation_correct = 0
        validation_total = 0

        with torch.no_grad():
            for data in validation_loader:
                inputs = data[0][probe.encoderLayer].to(device).float()
                targets = data[1].to(device)
                outputs = probe(inputs)
  
                loss = criterion(outputs, targets)
    
                validation_loss += loss.item()
                predicted = torch.argmax(outputs, dim=1)
                validation_total += targets.size(0)
                validation_correct += predicted.eq(targets).sum().item()

        validation_accuracy = validation_correct / validation_total
        writer.add_scalar(note + "/validation_accuracy", validation_accuracy, epoch)

        if validation_loss < best_validation_loss - 0.001:
            best_validation_loss = validation_loss
            best_model = copy.deepcopy(probe.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= 5:
            print("Early stopping...")
            break

    # probe is passed by reference, if it was improving for all epochs then we do not need to reload a previous state
    if best_model is not None:
        probe.load_state_dict(best_model)


def test(probe, loader, writer):
    labels_for_confusion = []
    predictions_for_confusion = []

    total = 0
    correct = 0

    probe.eval()

    for data in tqdm(
        loader,
        "Testing " + probe.encoder + " layer " + str(probe.encoderLayer),
        len(loader)
    ):
        inputs = data[0][probe.encoderLayer].to(device).float()
        targets = data[1].to(device)

        outputs = probe(inputs)

        predicted = torch.argmax(outputs, dim=1)
        total += predicted.size(0)
        correct += predicted.eq(targets).sum().item()

        cpu_target = targets.cpu()
        cpu_predicted = predicted.cpu()
        
        moa_targets = label_encoder.inverse_transform(cpu_target)
        moa_predicted = label_encoder.inverse_transform(cpu_predicted)

        labels_for_confusion.extend(moa_targets)
        predictions_for_confusion.extend(moa_predicted)
        
    accuracy = 100.*correct/total
    writer.add_scalar(probe.encoder + str(probe.encoderLayer) + ' test accuracy', accuracy)

    results_df = pd.DataFrame({
        "true_label": labels_for_confusion,
        "predicted_label": predictions_for_confusion
        })

    print("writing to " + "predictions/" + probe.encoder + "MOA/layer" + str(probe.encoderLayer) + ".csv")

    results_df.to_csv(
        "predictions/" + probe.encoder + "MOA/layer" + str(probe.encoderLayer) + ".csv",
        index=False
    )

def createConfusionFigure(encoder, encoderLayer):
    results_df = pd.read_csv("predictions/" + encoder + "MOA/layer" + str(encoderLayer) + ".csv")

    table_data = []

    for class_name in label_encoder.classes_:
        class_rows = results_df["true_label"] == class_name
        correct = results_df.loc[class_rows, "predicted_label"] == class_name

        class_accuracy = 100 * correct.sum() / class_rows.sum()
        table_data.append([class_name, f"{class_accuracy:.4f}%"])

    labels_for_confusion = results_df["true_label"].tolist()
    predictions_for_confusion = results_df["predicted_label"].tolist()

    fig, ax = plt.subplots(figsize=(10, 7))
    cm = confusion_matrix(labels_for_confusion, predictions_for_confusion, labels=label_encoder.classes_, normalize="true")

    ConfusionMatrixDisplay(cm, display_labels=label_encoder.classes_).plot(ax=ax)
    plt.title(encoder + " layer " + str(encoderLayer) + " Manner of Articulation")

    ax_table = fig.add_axes([0.85, 0.4, 0.15, 0.4])  # [left, bottom, width, height]
    ax_table.axis('off')
    table = ax_table.table(
        cellText=table_data,
        colLabels=['Class', 'Accuracy'],
        loc='center',
        cellLoc='center'
    )
    table.scale(1, 2)

    plt.savefig('figures/' + encoder + "/layer" + str(encoderLayer) + 'MOAconfusion.png', dpi=300, bbox_inches='tight')
    plt.close(fig)

## Whisper-small
----

### train whisper-small probes

In [5]:
if trainToggle:
    # load whisper train embeddings
    print("Loading saved whisper-small embeddings...")
    whisper_small_train_embeddings = torch.load('data/saved_embeddings/whisper_small_train_embeddings.pt')
    whisper_small_train_labels = torch.load('data/saved_embeddings/whisper_small_train_labels.pt')


Loading saved whisper-small embeddings...


In [7]:
if trainToggle:
    # set up whisper-small loader and label encoder
    print("Remapping + batching saved whisper-small embeddings....")
    whisper_small_train_loader, whisper_small_validate_loader = data_setup(whisper_small_train_embeddings, whisper_small_train_labels, labels_generated=False)
    torch.save(label_encoder, 'label_encoders/moa_label_encoder.pt') # data-setup has set up the label encoder that we will reuse

    for layerNumber in range(12):
        # train + save whisper probe per layer of embedding
        whisper_small_probe = ProbeNet(768, "whisper_small", layerNumber).to(device)
        optimizer = torch.optim.Adam(whisper_small_probe.parameters())
        criterion = nn.CrossEntropyLoss()

        writer = SummaryWriter(comment='whisper-small moa probe, layer ' + str(layerNumber))
        train(
            probe=whisper_small_probe,
            train_loader=whisper_small_train_loader,
            validation_loader=whisper_small_validate_loader,
            writer=writer,
            epochs=50,
            criterion=criterion,
            optimizer=optimizer,
            note='whisper-small moa probe accuracy, layer ' + str(layerNumber)
        )

        torch.save(whisper_small_probe, 'models/' + whisper_small_probe.encoder + 'MOA/layer' + str(whisper_small_probe.encoderLayer) + '.pth')

    # clear data that we're done with
    del whisper_small_train_embeddings
    del whisper_small_train_loader
    del whisper_small_train_labels

Remapping + batching saved whisper-small embeddings....
records: 128721
train batches: 805
validate batches: 202


Training whisper_small layer 0:  54%|█████▍    | 27/50 [01:13<01:02,  2.72s/it]


Early stopping...


Training whisper_small layer 1:  76%|███████▌  | 38/50 [01:41<00:32,  2.67s/it]


Early stopping...


Training whisper_small layer 2:  62%|██████▏   | 31/50 [01:23<00:51,  2.70s/it]


... (15 outputs left out)

Training whisper_small layer 10:  28%|██▊       | 14/50 [00:38<01:40,  2.78s/it]


Early stopping...


Training whisper_small layer 11:  34%|███▍      | 17/50 [00:46<01:31,  2.76s/it]

Early stopping...


### test probe + display confusion matrix

In [8]:
# whisper MOA test
print("Loading saved test whisper embeddings...")
whisper_small_test_embeddings = torch.load('data/saved_embeddings/whisper_small_test_embeddings.pt')
whisper_small_test_labels = torch.load('data/saved_embeddings/whisper_small_test_labels.pt')


Loading saved test whisper embeddings...


In [12]:
print("Remapping + batching saved test whisper-small embeddings....")
whisper_small_test_loader = data_setup(whisper_small_test_embeddings, whisper_small_test_labels, validation_frac=0.0)[0]

writer = SummaryWriter(comment='whisper-small moa probe')
label_encoder = torch.load('label_encoders/moa_label_encoder.pt', weights_only=False)

for layerNumber in range(12):
    encoder = 'whisper_small'
    whisper_probe = torch.load('models/' + encoder + 'MOA/layer' + str(layerNumber) + '.pth', weights_only=False)

    whisper_probe.eval()
    test(whisper_probe, whisper_small_test_loader, writer)

    createConfusionFigure("whisper_small", layerNumber)

del whisper_small_test_embeddings
del whisper_small_test_labels

Remapping + batching saved test whisper-small embeddings....
records: 47013
train batches: 368
validate batches: 0


Testing whisper_small layer 0: 100%|██████████| 368/368 [00:01<00:00, 358.31it/s]


writing to predictions/whisper_smallMOA/layer0.csv


Testing whisper_small layer 1: 100%|██████████| 368/368 [00:01<00:00, 347.27it/s]


writing to predictions/whisper_smallMOA/layer1.csv


Testing whisper_small layer 2: 100%|██████████| 368/368 [00:00<00:00, 384.49it/s]


... (15 outputs left out)

Testing whisper_small layer 10: 100%|██████████| 368/368 [00:00<00:00, 379.13it/s]


writing to predictions/whisper_smallMOA/layer10.csv


Testing whisper_small layer 11: 100%|██████████| 368/368 [00:01<00:00, 364.80it/s]


writing to predictions/whisper_smallMOA/layer11.csv


### Whisper-large-v3

--- 

train whisper-large probes

In [5]:
if trainToggle:
    # load whisper train embeddings
    print("Loading saved whisper-large embeddings...")
    whisper_large_train_embeddings = torch.load('data/saved_embeddings/whisper_large_train_embeddings.pt')
    whisper_large_train_labels = torch.load('data/saved_embeddings/whisper_large_train_labels.pt')


Loading saved whisper-large embeddings...


In [6]:
if trainToggle:
    # set up whisper-large loader and label encoder
    print("Remapping + batching saved whisper-large embeddings....")
    whisper_large_train_loader, whisper_large_validate_loader = data_setup(whisper_large_train_embeddings, whisper_large_train_labels, labels_generated=False)
    torch.save(label_encoder, 'label_encoders/moa_label_encoder.pt') # data-setup has set up the label encoder that we will reuse

    for layerNumber in range(32):
        # train + save whisper probe per layer of embedding
        whisper_large_probe = ProbeNet(1280, "whisper_large", layerNumber).to(device)
        optimizer = torch.optim.Adam(whisper_large_probe.parameters())
        criterion = nn.CrossEntropyLoss()

        writer = SummaryWriter(comment='whisper-large moa probe, layer ' + str(layerNumber))
        train(
            probe=whisper_large_probe,
            train_loader=whisper_large_train_loader,
            validation_loader=whisper_large_validate_loader,
            writer=writer,
            epochs=100,
            criterion=criterion,
            optimizer=optimizer,
            note='whisper-large moa probe accuracy, layer ' + str(layerNumber)
        )

        torch.save(whisper_large_probe, 'models/' + whisper_large_probe.encoder + 'MOA/layer' + str(whisper_large_probe.encoderLayer) + '.pth')

    # clear data that we're done with
    del whisper_large_train_embeddings
    del whisper_large_train_loader
    del whisper_large_train_labels

Remapping + batching saved whisper-large embeddings....
records: 128721
train batches: 805
validate batches: 202


Training whisper_large layer 0:  27%|██▋       | 27/100 [02:13<06:01,  4.95s/it]


Early stopping...


Training whisper_large layer 1:  17%|█▋        | 17/100 [01:25<06:56,  5.02s/it]


Early stopping...


Training whisper_large layer 2:  30%|███       | 30/100 [02:27<05:43,  4.91s/it]


... (55 outputs left out)

Training whisper_large layer 30:  34%|███▍      | 34/100 [02:44<05:19,  4.84s/it]


Early stopping...


Training whisper_large layer 31:  36%|███▌      | 36/100 [02:54<05:10,  4.84s/it]

Early stopping...


### test probe + display confusion matrix

In [7]:
# whisper MOA test
print("Loading saved test whisper embeddings...")
whisper_large_test_embeddings = torch.load('data/saved_embeddings/whisper_large_test_embeddings.pt')
whisper_large_test_labels = torch.load('data/saved_embeddings/whisper_large_test_labels.pt')


Loading saved test whisper embeddings...


In [8]:
print("Remapping + batching saved test whisper-large embeddings....")
whisper_large_test_loader = data_setup(whisper_large_test_embeddings, whisper_large_test_labels, validation_frac=0.0)[0]

writer = SummaryWriter(comment='whisper-large moa probe')
label_encoder = torch.load('label_encoders/moa_label_encoder.pt', weights_only=False)

for layerNumber in range(32):
    encoder = 'whisper_large'
    whisper_probe = torch.load('models/' + encoder + 'MOA/layer' + str(layerNumber) + '.pth', weights_only=False)

    whisper_probe.eval()
    test(whisper_probe, whisper_large_test_loader, writer)

    createConfusionFigure("whisper_large", layerNumber)

del whisper_large_test_embeddings
del whisper_large_test_labels

Remapping + batching saved test whisper-large embeddings....
records: 47013
train batches: 368
validate batches: 0


Testing whisper_large layer 0: 100%|██████████| 368/368 [00:01<00:00, 186.68it/s]


writing to predictions/whisper_largeMOA/layer0.csv


Testing whisper_large layer 1: 100%|██████████| 368/368 [00:01<00:00, 184.90it/s]


writing to predictions/whisper_largeMOA/layer1.csv


Testing whisper_large layer 2: 100%|██████████| 368/368 [00:01<00:00, 196.57it/s]


... (55 outputs left out)

Testing whisper_large layer 30: 100%|██████████| 368/368 [00:01<00:00, 207.02it/s]


writing to predictions/whisper_largeMOA/layer30.csv


Testing whisper_large layer 31: 100%|██████████| 368/368 [00:01<00:00, 213.91it/s]


writing to predictions/whisper_largeMOA/layer31.csv


## wav2vec
---
### train wav2vec

In [17]:
# initialize and train the wav2vec probe
if trainToggle:
    print("Loading saved wav2vec embeddings...")
    wav2vec_train_embeddings = torch.load('data/saved_embeddings/wav2vec_train_embeddings.pt')
    wav2vec_train_labels = torch.load('data/saved_embeddings/wav2vec_train_labels.pt')


Loading saved wav2vec embeddings...


In [18]:
if trainToggle:
    # set up train loader for wav2vec
    print("Remapping + batching saved wav2vec embeddings....")
    wav2vec_train_loader, wav2vec_validate_loader = data_setup(wav2vec_train_embeddings, wav2vec_train_labels, labels_generated=False)

    for layerNumber in range(12):
        # train + save probe per layer of embedding
        wav2vec_probe = ProbeNet(768, "wav2vec", layerNumber).to(device)
        optimizer = torch.optim.Adam(wav2vec_probe.parameters())
        criterion = nn.CrossEntropyLoss()

        writer = SummaryWriter(comment='wav2vec probe, layer ' + str(layerNumber))
        train(
            probe=wav2vec_probe,
            train_loader=wav2vec_train_loader,
            validation_loader=wav2vec_validate_loader,
            writer=writer,
            epochs=50,
            criterion=criterion,
            optimizer=optimizer,
            note='wav2vec moa probe accuracy, layer ' + str(layerNumber)
        )

        torch.save(wav2vec_probe, 'models/' + wav2vec_probe.encoder + 'MOA/layer' + str(wav2vec_probe.encoderLayer) + '.pth')


    # clear data that we're done with
    del wav2vec_train_embeddings
    del wav2vec_train_loader
    del wav2vec_train_labels

Remapping + batching saved wav2vec embeddings....
records: 128721
train batches: 805
validate batches: 202


Training wav2vec layer 0:  16%|█▌        | 8/50 [00:24<02:09,  3.09s/it]


Early stopping...


Training wav2vec layer 1:  32%|███▏      | 16/50 [00:45<01:35,  2.82s/it]


Early stopping...


Training wav2vec layer 2:  26%|██▌       | 13/50 [00:37<01:46,  2.88s/it]


... (15 outputs left out)

Training wav2vec layer 10:  44%|████▍     | 22/50 [01:01<01:17,  2.78s/it]


Early stopping...


Training wav2vec layer 11:  64%|██████▍   | 32/50 [01:27<00:49,  2.74s/it]

Early stopping...


### test wav2vec + display confusion matrix

In [19]:
# test wav2vec MOA probe
print("Loading saved test wav2vec embeddings...")
wav2vec_test_embeddings = torch.load('data/saved_embeddings/wav2vec_test_embeddings.pt')
wav2vec_test_labels = torch.load('data/saved_embeddings/wav2vec_test_labels.pt')


Loading saved test wav2vec embeddings...


In [20]:
print("Remapping + batching saved test wav2vec embeddings....")
wav2vec_test_loader = data_setup(wav2vec_test_embeddings, wav2vec_test_labels, validation_frac=0.0)[0]

writer = SummaryWriter(comment='wav2vec moa probe')
label_encoder = torch.load('label_encoders/moa_label_encoder.pt', weights_only=False)

for layerNumber in range(12):
    encoder = 'wav2vec'
    wav2vec_probe = torch.load('models/' + encoder + 'MOA/layer' + str(layerNumber) + '.pth', weights_only=False)

    wav2vec_probe.eval()
    test(wav2vec_probe, wav2vec_test_loader, writer)

    createConfusionFigure("wav2vec", layerNumber)

del wav2vec_test_embeddings
del wav2vec_test_labels

Remapping + batching saved test wav2vec embeddings....
records: 47013
train batches: 368
validate batches: 0


Testing wav2vec layer 0: 100%|██████████| 368/368 [00:00<00:00, 371.43it/s]


writing to predictions/wav2vecMOA/layer0.csv


Testing wav2vec layer 1: 100%|██████████| 368/368 [00:00<00:00, 399.01it/s]


writing to predictions/wav2vecMOA/layer1.csv


Testing wav2vec layer 2: 100%|██████████| 368/368 [00:00<00:00, 387.48it/s]


... (15 outputs left out)

Testing wav2vec layer 10: 100%|██████████| 368/368 [00:00<00:00, 417.13it/s]


writing to predictions/wav2vecMOA/layer10.csv


Testing wav2vec layer 11: 100%|██████████| 368/368 [00:00<00:00, 429.79it/s]


writing to predictions/wav2vecMOA/layer11.csv


## voxtral
---
### train

In [21]:
if trainToggle:
    # initialize and train the voxtral probe
    print("Loading saved voxtral embeddings...")
    voxtral_train_embeddings = torch.load('data/saved_embeddings/voxtral_train_embeddings.pt')
    voxtral_train_labels = torch.load('data/saved_embeddings/voxtral_train_labels.pt')


Loading saved voxtral embeddings...


In [22]:
if trainToggle:
    # set up voxtral loader
    print("Remapping + batching saved voxtral embeddings....")
    voxtral_train_loader, voxtral_validate_loader = data_setup(voxtral_train_embeddings, voxtral_train_labels, labels_generated=False)

    for layerNumber in range(32):
        # train + save probe per layer
        voxtral_probe = ProbeNet(1280, "voxtral", layerNumber).to(device)
        optimizer = torch.optim.Adam(voxtral_probe.parameters())
        criterion = nn.CrossEntropyLoss()

        writer = SummaryWriter(comment='voxtral probe, layer ' + str(layerNumber))
        train(
            probe=voxtral_probe,
            train_loader=voxtral_train_loader,
            validation_loader=voxtral_validate_loader,
            writer=writer,
            epochs=50,
            criterion=criterion,
            optimizer=optimizer,
            note='voxtral moa probe accuracy, layer ' + str(layerNumber)
        )

        torch.save(voxtral_probe, 'models/' + voxtral_probe.encoder + 'MOA/layer' + str(voxtral_probe.encoderLayer) + '.pth')

    # clear data that we're done with
    del voxtral_train_embeddings
    del voxtral_train_loader
    del voxtral_train_labels

Remapping + batching saved voxtral embeddings....
records: 128721
train batches: 805
validate batches: 202


Training voxtral layer 0:  72%|███████▏  | 36/50 [02:51<01:06,  4.77s/it]


Early stopping...


Training voxtral layer 1:  60%|██████    | 30/50 [02:22<01:35,  4.77s/it]


Early stopping...


Training voxtral layer 2:  46%|████▌     | 23/50 [01:50<02:10,  4.82s/it]


... (55 outputs left out)

Training voxtral layer 30:  28%|██▊       | 14/50 [01:09<02:58,  4.96s/it]


Early stopping...


Training voxtral layer 31:  68%|██████▊   | 34/50 [02:42<01:16,  4.77s/it]

Early stopping...


### test + confusion matrix

In [23]:
# voxtral MOA test
print("Loading saved test voxtral embeddings...")
voxtral_test_embeddings = torch.load('data/saved_embeddings/voxtral_test_embeddings.pt')
voxtral_test_labels = torch.load('data/saved_embeddings/voxtral_test_labels.pt')


Loading saved test voxtral embeddings...


In [ ]:

print("Remapping + batching saved test voxtral embeddings....")
voxtral_test_loader = data_setup(voxtral_test_embeddings, voxtral_test_labels, validation_frac=0.0)[0]

writer = SummaryWriter(comment='voxtral moa probe')
label_encoder = torch.load('label_encoders/moa_label_encoder.pt', weights_only=False)

for layerNumber in range(32):
    encoder = 'voxtral'
    voxtral_probe = torch.load('models/' + encoder + 'MOA/layer' + str(layerNumber) + '.pth', weights_only=False)

    voxtral_probe.eval()

    test(voxtral_probe, voxtral_test_loader, writer)

    createConfusionFigure("voxtral", layerNumber)

Remapping + batching saved test voxtral embeddings....
records: 47013
train batches: 368
validate batches: 0


Testing voxtral layer 0: 100%|██████████| 368/368 [00:02<00:00, 156.00it/s]


writing to predictions/voxtralMOA/layer0.csv


Testing voxtral layer 1: 100%|██████████| 368/368 [00:01<00:00, 189.53it/s]


writing to predictions/voxtralMOA/layer1.csv


Testing voxtral layer 2: 100%|██████████| 368/368 [00:01<00:00, 194.25it/s]


... (56 outputs left out)

writing to predictions/voxtralMOA/layer30.csv


Testing voxtral layer 31: 100%|██████████| 368/368 [00:01<00:00, 223.82it/s]


writing to predictions/voxtralMOA/layer31.csv


: 